In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random
from full_model import predict
from analyte import ANALYTES


con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT id, age, gender, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data 
                 WHERE value IS NOT NULL
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()

pids = con.execute("SELECT DISTINCT pid FROM protein_data").df()


con.close()

df = df[
    (df['fractions'].apply(len) == 6) &
    (df['boundaries'].apply(len) == 12)
]

protein_cols = [a.col for a in ANALYTES[:8]]  # bara de 8 första
df = df.dropna(subset=protein_cols)
y = df['label']

test_rows = df[df['set'] == 'test']
processed_rows = predict(test_rows['id'],test_rows)

Laddar modeller...


In [2]:
import importlib
import full_model
importlib.reload(full_model)
from full_model import interpret

idx = random.randrange(0,len(test_rows) -1 ,1)
#rows = processed_rows[processed_rows['proportion_gamma_region'] > 0.9].sort_values('proportion_gamma_region',ascending=False)
rows = processed_rows[(processed_rows['cnn_probability'] < 0.2) & (processed_rows['label'] == 1)]
row = rows.iloc[2,:]
#row = test_rows.iloc[idx,:]
result = interpret(row.to_dict())

print(len(test_rows))

Laddar modeller...


KeyError: 'gender'